In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# ==========================================
# 1. 
# ==========================================
MERGED_DATA = '../../Dataset/data/merged_data_40501_updated.json'

print(f"Loading data from: {MERGED_DATA}")
with open(MERGED_DATA, 'r', encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)

target_col = "is_self_fixed"

# 
#  X， y 
y = df[target_col].astype(int).values  #  int (0/1)
groups = df["project_name"].values     #  LOPO
X = np.zeros((len(y), 1))              #  (Dummy X)

print(f"Data Loaded. Samples: {len(y)}, Positive Ratio: {y.mean():.2%}")



Loading data from: ../../Dataset/data/merged_data_40501_updated.json
Data Loaded. Samples: 40501, Positive Ratio: 57.28%


In [ ]:
# def random_guess_predict(y_true):
#     n = len(y_true)
    
#     # 1.  AUC
#     y_prob = np.random.rand(n)
    
#     # 2.  0/1  ()
#     # ：， seed，
#     # ， np.random.seed
#     y_pred = np.random.randint(0, 2, n) 
    
#     return y_prob, y_pred

def random_guess_predict(y_true):
    """
    Stratified Random Guess:
    。
     20%， 20% 。
    """
    n = len(y_true)
    
    # 1.  (Prior)
    # ，，
    #  Baseline，。
    pos_ratio = np.mean(y_true) 
    
    # 2. 
    y_prob = np.random.rand(n)
    
    # 3. 
    #  Top X%  (X = pos_ratio)
    n_pos = int(n * pos_ratio)
    
    sorted_indices = np.argsort(y_prob)
    y_pred = np.zeros(n)
    if n_pos > 0:
        y_pred[sorted_indices[-n_pos:]] = 1
        
    return y_prob, y_pred

In [ ]:
def get_metrics(y_true, y_prob, y_pred):
    if len(np.unique(y_true)) < 2:
        auc = 0.5 
    else:
        auc = roc_auc_score(y_true, y_prob)
        
    return {
        'AUC': auc,
        'ACC': accuracy_score(y_true, y_pred),
        'Prec': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# ==========================================
# 3.  10 
# ==========================================
print("\nRunning Task 1: Stratified 10-Fold CV...")

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    y_test = y[test_idx]
    r_prob, r_pred = random_guess_predict(y_test)
    
    metrics = get_metrics(y_test, r_prob, r_pred)
    metrics['Fold'] = fold + 1
    cv_results.append(metrics)

df_cv = pd.DataFrame(cv_results)
cv_mean = df_cv.mean(numeric_only=True)

# ==========================================
# 4.  LOPO 
# ==========================================
print("Running Task 2: LOPO Cross-Project...")

logo = LeaveOneGroupOut()
lopo_results = []

for train_idx, test_idx in logo.split(X, y, groups=groups):
    y_test = y[test_idx]
    current_project = groups[test_idx][0]
    
    if len(y_test) < 2: continue
        
    r_prob, r_pred = random_guess_predict(y_test)
    
    metrics = get_metrics(y_test, r_prob, r_pred)
    metrics['Project'] = current_project
    lopo_results.append(metrics)

df_lopo = pd.DataFrame(lopo_results)
lopo_mean = df_lopo.mean(numeric_only=True)

# ==========================================
# 5.  ()
# ==========================================

# ---  10-Fold CV ---
cv_mean_row = cv_mean.to_frame().T
cv_mean_row['Fold'] = 'Average'
df_cv_final = pd.concat([df_cv, cv_mean_row], ignore_index=True)

# 【】：， Fold 
cols_cv = ['Fold', 'AUC', 'ACC', 'Prec', 'Recall', 'F1']
df_cv_final = df_cv_final[cols_cv]

df_cv_final.to_csv('random_guess_10fold.csv', index=False)
print(f"Saved: random_guess_10fold.csv")


# ---  LOPO ---
lopo_mean_row = lopo_mean.to_frame().T
lopo_mean_row['Project'] = 'Average'
df_lopo_final = pd.concat([df_lopo, lopo_mean_row], ignore_index=True)

# 【】：， Project 
cols_lopo = ['Project', 'AUC', 'ACC', 'Prec', 'Recall', 'F1']
df_lopo_final = df_lopo_final[cols_lopo]

df_lopo_final.to_csv('random_guess_lopo.csv', index=False)
print(f"Saved: random_guess_lopo.csv")

# 
print("\nPreview (10-Fold):")
print(df_cv_final.tail(2))


Running Task 1: Stratified 10-Fold CV...
Running Task 2: LOPO Cross-Project...
Saved: random_guess_10fold.csv
Saved: random_guess_lopo.csv

Preview (10-Fold):
       Fold       AUC       ACC      Prec    Recall        F1
9        10  0.510267  0.516049  0.577404  0.577404  0.577404
10  Average  0.500104  0.510876  0.573042  0.573042  0.573042


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# 【】：，
np.random.seed(42)

# ==========================================
# 1. 
# ==========================================
MERGED_DATA = '../../Dataset/data/merged_data_40501_updated.json'

print(f"Loading data from: {MERGED_DATA}")
with open(MERGED_DATA, 'r', encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)

target_col = "is_self_fixed"
y = df[target_col].astype(int).values
groups = df["project_name"].values
X = np.zeros((len(y), 1)) # 

print(f"Data Loaded. Samples: {len(y)}, Positive Ratio: {y.mean():.2%}")

# ==========================================
# 2.  ()
# ==========================================

def predict_uniform(y_true):
    """
     A: Uniform Random Guess (50/50)
    ， 50% 。
    """
    n = len(y_true)
    y_prob = np.random.rand(n) #  seed(42) 
    n_pos = int(n * 0.5) 
    
    sorted_indices = np.argsort(y_prob)
    y_pred = np.zeros(n)
    if n_pos > 0:
        y_pred[sorted_indices[-n_pos:]] = 1
    return y_prob, y_pred

def predict_stratified(y_true, train_pos_ratio):
    """
     B: Stratified Random Guess ()
    【】： y_true ， train_pos_ratio ()。
    ：()，。
    """
    n = len(y_true)
    y_prob = np.random.rand(n) #  seed(42) 
    
    # 1
    n_pos = int(n * train_pos_ratio)
    
    sorted_indices = np.argsort(y_prob)
    y_pred = np.zeros(n)
    if n_pos > 0:
        y_pred[sorted_indices[-n_pos:]] = 1
    return y_prob, y_pred

# 
def get_metrics(y_true, y_prob, y_pred):
    if len(np.unique(y_true)) < 2:
        auc = 0.5 
    else:
        auc = roc_auc_score(y_true, y_prob)
        
    return {
        'AUC': auc,
        'ACC': accuracy_score(y_true, y_pred),
        'Prec': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# 
def save_results(results_list, filename, id_col_name):
    df_res = pd.DataFrame(results_list)
    
    # 
    mean_row = df_res.mean(numeric_only=True).to_frame().T
    mean_row[id_col_name] = 'Average'
    
    # 
    df_final = pd.concat([df_res, mean_row], ignore_index=True)
    
    # 
    cols = [id_col_name, 'AUC', 'ACC', 'Prec', 'Recall', 'F1']
    df_final = df_final[cols]
    
    df_final.to_csv(filename, index=False)
    print(f"Saved: {filename}")

# ==========================================
# 3.  10 
# ==========================================
print("\n" + "="*40)
print("Task 1: Stratified 10-Fold CV")
print("="*40)

res_cv_uniform = []
res_cv_stratified = []

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    y_train, y_test = y[train_idx], y[test_idx]
    
    # --- 1.  Uniform (50/50) ---
    u_prob, u_pred = predict_uniform(y_test)
    m_u = get_metrics(y_test, u_prob, u_pred)
    m_u['Fold'] = fold + 1
    res_cv_uniform.append(m_u)
    
    # --- 2.  Stratified () ---
    # []  (y_train) 
    train_ratio = np.mean(y_train) 
    
    s_prob, s_pred = predict_stratified(y_test, train_ratio) # 
    m_s = get_metrics(y_test, s_prob, s_pred)
    m_s['Fold'] = fold + 1
    res_cv_stratified.append(m_s)

save_results(res_cv_uniform, 'random_guess_uniform_10fold.csv', 'Fold')
save_results(res_cv_stratified, 'random_guess_stratified_10fold.csv', 'Fold')

# ==========================================
# 4.  LOPO 
# ==========================================
print("\n" + "="*40)
print("Task 2: LOPO Cross-Project")
print("="*40)

res_lopo_uniform = []
res_lopo_stratified = []

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups=groups):
    y_train, y_test = y[train_idx], y[test_idx]
    current_project = groups[test_idx][0]
    
    if len(y_test) < 2: continue
        
    # --- 1.  Uniform (50/50) ---
    u_prob, u_pred = predict_uniform(y_test)
    m_u = get_metrics(y_test, u_prob, u_pred)
    m_u['Project'] = current_project
    res_lopo_uniform.append(m_u)
    
    # --- 2.  Stratified () ---
    # []  (y_train)  ()
    train_ratio = np.mean(y_train)
    
    s_prob, s_pred = predict_stratified(y_test, train_ratio) # 
    m_s = get_metrics(y_test, s_prob, s_pred)
    m_s['Project'] = current_project
    res_lopo_stratified.append(m_s)

save_results(res_lopo_uniform, 'random_guess_uniform_lopo.csv', 'Project')
save_results(res_lopo_stratified, 'random_guess_stratified_lopo.csv', 'Project')

print("\nAll tasks completed. 4 CSV files generated.")

Loading data from: ../../Dataset/data/merged_data_40501_updated.json
Data Loaded. Samples: 40501, Positive Ratio: 57.28%

Task 1: Stratified 10-Fold CV
Saved: random_guess_uniform_10fold.csv
Saved: random_guess_stratified_10fold.csv

Task 2: LOPO Cross-Project
Saved: random_guess_uniform_lopo.csv
Saved: random_guess_stratified_lopo.csv

All tasks completed. 4 CSV files generated.
